# 04 — Local Delta governance (OSS)

Replace Unity Catalog / Autoloader sketches with checks against local Delta tables: contract columns, Bronze provenance presence, and row counts.


In [ ]:
from pathlib import Path
from ambient_pipeline.notebook_bootstrap import ensure_pipeline_on_path, apply_spark_tuning

ROOT = ensure_pipeline_on_path(Path.cwd())
assert ROOT is not None, "Run from ambient-core checkout (lib/ambient_pipeline missing)"
print(f"repo root: {ROOT}")


In [ ]:
from ambient_pipeline.perf import create_local_spark

spark = create_local_spark(app_name="ambient-oss-notebooks", shuffle_partitions=4)
apply_spark_tuning(spark)
print(spark.version)


In [ ]:
from ambient_pipeline.contracts import ContractLoader
from ambient_pipeline.storage_paths import resolve_table_path

out_base = str(ROOT / ".lakehouse" / "demo")
silver_table = resolve_table_path("local", out_base, "demo", "silver", "tenant_metrics")
silver = spark.read.format("delta").load(silver_table)

loader = ContractLoader()
contract = loader.load("tenant-metrics-v1.yaml")
loader.enforce_bronze_lineage(contract)
loader.assert_required_columns(set(silver.columns), contract, "local_delta_governance")

required_bronze = {"_bronze_run_id", "_bronze_org_id", "_bronze_row_hash", "_bronze_ingestion_ts"}
missing = required_bronze - set(silver.columns)
assert not missing, missing

print("contract + bronze provenance: ok")
print(f"silver rows={silver.count()}")
print("columns:", sorted(silver.columns)[:20], "...")
